# 🤖 Notebook 03: Predictive Machine Learning & Customer Churn Modeling
### European Banking | Unified Mentor Internship | ECB Analytics
**Author:** Rishi Prajapati  
**Target:** Predict customer attrition (`Exited`), isolate risk determinants, and optimize retention intervention thresholds.

---
### 📌 Executive Summary
While exploratory data analysis (Notebook 01 & 02) revealed aggregate patterns—such as elevated German customer attrition and high churn among clients aged 46–60—traditional descriptive analytics cannot quantify non-linear multi-factor interactions or score incoming customers in real time.

In this notebook, we develop an **enterprise-grade Machine Learning classification suite**:
1. **Domain Feature Engineering**: Synthesize banking capital ratios, tenure-age loyalty indices, and behavioral interaction flags.
2. **Modular Preprocessing**: Build scikit-learn `ColumnTransformer` pipelines with scaling and categorical encoding.
3. **Model Benchmarking**: Compare **Logistic Regression**, **Random Forest**, **Gradient Boosting**, and **XGBoost Classifier** under 5-Fold Stratified Cross-Validation.
4. **Imbalance & Cost Handling**: Address the ~80:20 class imbalance using positive weighting and threshold optimization to minimize costly False Negatives.
5. **Interpretability & Production Deployment**: Rank global risk drivers and export the best serialization artifact for the Streamlit dashboard.


In [ ]:
# ── Core Dependencies ───────────────────────────────────────────────
import sys
import os
from pathlib import Path

# Ensure root directory is accessible
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix,
    classification_report
)
import xgboost as xgb

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Arial'
print("✅ Libraries and environment successfully initialized.")


In [ ]:
# ── Load Dataset ──────────────────────────────────────────────────
data_path = ROOT_DIR / 'data' / 'processed' / 'churn_final_scored.csv'
if not data_path.exists():
    data_path = ROOT_DIR / 'data' / 'raw' / 'Churn_Modelling.csv'

df = pd.read_csv(data_path)
print(f'Dataset Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('Target Distribution (Exited):')
print(df['Exited'].value_counts(normalize=True).round(4) * 100)
df.head(5)


## 🛠️ Feature Engineering & Preprocessing Architecture

We segment features into numerical and categorical subsets, integrating domain ratios:
- **Numerical**: `CreditScore`, `Age`, `Tenure`, `Balance`, `NumOfProducts`, `EstimatedSalary`, `BalanceSalaryRatio`, `TenureAgeRatio`, `CreditScoreAgeRatio`, `ProductsPerTenure`
- **Categorical**: `Geography`, `Gender`, `HasCrCard`, `IsActiveMember`, `IsZeroBalance`


In [ ]:
# ── Feature Definition ───────────────────────────────────────────
num_cols = [
    'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
    'EstimatedSalary', 'BalanceSalaryRatio', 'TenureAgeRatio',
    'CreditScoreAgeRatio', 'ProductsPerTenure'
]

cat_cols = ['Geography', 'Gender', 'HasCrCard', 'IsActiveMember', 'IsZeroBalance']
target_col = 'Exited'

X = df[num_cols + cat_cols].copy()
y = df[target_col].copy()

# Stratified 80/20 train-test partition
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train split: {X_train.shape[0]:,} samples | Test split: {X_test.shape[0]:,} samples")

# Build Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)
print("✅ ColumnTransformer pipeline configured.")


## 🔬 Multi-Model Benchmark & Cross-Validation

We evaluate 4 distinct algorithms covering linear, bagging, and boosting paradigms:
1. **Logistic Regression** (L2 penalty, balanced weights)
2. **Random Forest** (Ensemble of decorrelated decision trees)
3. **Gradient Boosting** (Sequential gradient error correction)
4. **XGBoost Classifier** (Optimized extreme gradient boosting with scale_pos_weight=3.2)


In [ ]:
# ── Model Suite Definition ──────────────────────────────────────
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, max_depth=12, min_samples_split=6,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.08, max_depth=4, subsample=0.85, random_state=42
    ),
    "XGBoost Classifier": xgb.XGBClassifier(
        n_estimators=220, learning_rate=0.07, max_depth=4, subsample=0.85,
        colsample_bytree=0.85, scale_pos_weight=3.2, eval_metric='logloss', random_state=42
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
benchmark_records = []
fitted_models = {}
probas_dict = {}

print("🚀 Running 5-Fold Cross Validation and Holdout Evaluation...")
for name, clf in models.items():
    pipe = Pipeline([
        ('prep', preprocessor),
        ('clf', clf)
    ])
    
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    probas_dict[name] = y_prob
    
    benchmark_records.append({
        "Model": name,
        "CV ROC-AUC (Mean)": round(float(np.mean(cv_scores)), 4),
        "CV ROC-AUC (Std)": round(float(np.std(cv_scores)), 4),
        "Test ROC-AUC": round(roc_auc_score(y_test, y_prob), 4),
        "Test Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1-Score": round(f1_score(y_test, y_pred, zero_division=0), 4)
    })

benchmark_df = pd.DataFrame(benchmark_records).sort_values("Test ROC-AUC", ascending=False)
benchmark_df


## 📈 ROC & Precision-Recall Performance Curves


In [ ]:
# ── Plot ROC and Precision-Recall Curves ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

colors = {'Logistic Regression': '#6B7280', 'Random Forest': '#3B82F6', 
          'Gradient Boosting': '#10B981', 'XGBoost Classifier': '#EF4444'}

# ROC Curve
for name, y_prob in probas_dict.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.3f})", color=colors[name], lw=2)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.6, label='Random Chance')
axes[0].set_title("Receiver Operating Characteristic (ROC) Curves", fontsize=14, weight='bold')
axes[0].set_xlabel("False Positive Rate", fontsize=12)
axes[0].set_ylabel("True Positive Rate (Recall)", fontsize=12)
axes[0].legend(loc="lower right")

# Precision-Recall Curve
for name, y_prob in probas_dict.items():
    p, r, _ = precision_recall_curve(y_test, y_prob)
    axes[1].plot(r, p, label=name, color=colors[name], lw=2)

axes[1].set_title("Precision-Recall Curves", fontsize=14, weight='bold')
axes[1].set_xlabel("Recall", fontsize=12)
axes[1].set_ylabel("Precision", fontsize=12)
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()


## ⚖️ Confusion Matrix & Financial Cost Matrix Analysis

In retail banking, a **False Negative** (failing to detect an exiting customer) leads to lost customer lifetime value (LTV ≈ €2,500), whereas a **False Positive** (reaching out with retention incentives to a stable customer) incurs minor administrative cost (~€150). Therefore, optimizing for higher Recall is critical.


In [ ]:
# ── Confusion Matrix for Best Model (XGBoost) ─────────────────
best_name = "XGBoost Classifier"
best_pipe = fitted_models[best_name]
y_pred_best = best_pipe.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Retained', 'Predicted Churned'],
            yticklabels=['Actual Retained', 'Actual Churned'], ax=ax)
ax.set_title(f"Confusion Matrix: {best_name}", fontsize=13, weight='bold')
plt.show()

# Financial impact estimate
cost_fn = 2500  # Lost LTV
cost_fp = 150   # Retention outreach
total_cost = cm[1, 0] * cost_fn + cm[0, 1] * cost_fp
print(f"Calculated Financial Risk Exposure: €{total_cost:,.2f}")


## 🔍 Explainability & Feature Importance Ranking


In [ ]:
# ── Extract Feature Importances ────────────────────────────────
clf_step = best_pipe.named_steps['clf']
prep_step = best_pipe.named_steps['prep']

cat_encoder = prep_step.named_transformers_['cat']
cat_names = list(cat_encoder.get_feature_names_out(cat_cols))
all_feat_names = num_cols + cat_names

importances = clf_step.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': all_feat_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp.head(10), x='Importance', y='Feature', palette='Blues_r')
plt.title("Top 10 Churn Predictors (XGBoost Feature Importance)", fontsize=14, weight='bold')
plt.xlabel("Normalized Importance Score")
plt.tight_layout()
plt.show()


## 🧪 Real-Time Customer Inference & Counterfactual Simulation


In [ ]:
# ── Live Customer Scoring Demonstration ─────────────────────────
from src.inference import ChurnPredictor, SAMPLE_PERSONAS

predictor = ChurnPredictor()
demo_persona = SAMPLE_PERSONAS['🇩🇪 High-Risk German VIP (Age 52, High Balance, Inactive)']

prediction_output = predictor.predict(demo_persona)
print("=== Real-Time Scoring Output ===")
print(f"Customer Risk Tier     : {prediction_output['badge']}")
print(f"Calculated Churn Risk  : {prediction_output['churn_probability_pct']}%")
print(f"Identified Drivers     : {prediction_output['risk_drivers']}")
print(f"Retention Actions      : {prediction_output['recommended_actions']}")

# Counterfactual: What if this customer activates their account?
simulation = predictor.simulate_what_if(demo_persona, {'IsActiveMember': 1, 'NumOfProducts': 2})
print(f"\nCounterfactual Result : Churn drops from {simulation['base_pct']}% -> {simulation['new_pct']}% (Risk Delta: {simulation['delta_pct']}%)")


## 🎯 Strategic Recommendations & Next Steps

1. **Deploy Model Artifact**: Pipeline is serialized to `models/churn_pipeline.joblib` for zero-latency inference on Streamlit.
2. **Proactive Intervention Trigger**: Customers exceeding **45% churn probability** should be automatically routed to CRM retention queues.
3. **Cross-Sell Focus**: Transition customers with 1 product or 3+ products into the sweet spot of **2 products**, which exhibits the lowest empirical churn rate (~7.6%).
4. **Active Engagement Campaigns**: Inactivity is the largest single actionable driver—targeted mobile app engagement campaigns can yield a 30%+ drop in attrition probability.
